# 01 - Dataset Acquisition dan EDA (SisaRasa, Phase 1)

Notebook ini hanya melakukan Phase 1: memeriksa dataset. Tidak ada preprocessing destruktif, normalisasi bahan, retrieval, atau model.

Aturan notebook ini:
- Data sumber tidak diubah. Semua kolom turunan memakai awalan `_eda_` dan hanya hidup di salinan.
- Tidak ada angka yang ditulis tangan. Semua angka berasal dari data yang benar-benar dimuat.
- Bagian kesimpulan dan lisensi di akhir harus kamu isi sendiri setelah membaca hasil dan halaman sumbernya.

## 0. Cek environment dan file input

In [ ]:
import sys, os, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    import psutil
    print("RAM total (GB):", round(psutil.virtual_memory().total / 1e9, 1))
except ImportError:
    print("psutil tidak tersedia")
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada (CPU saja)")
except ImportError:
    print("torch tidak terpasang")

INPUT_DIR = os.environ.get("INPUT_DIR", "/kaggle/input")
print("\nIsi", INPUT_DIR)
n_files = 0
for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        p = os.path.join(root, f)
        print(" ", p, "|", round(os.path.getsize(p) / 1e6, 2), "MB")
        n_files += 1
if n_files == 0:
    print("  (kosong) -> tambahkan dataset lewat Add Input, atau pakai SOURCE = 'hf' di cell berikutnya")

## 1. Konfigurasi

In [ ]:
# Ubah hanya bagian ini bila perlu.
SOURCE = "kaggle"                      # "kaggle" (dari /kaggle/input) atau "hf" (Hugging Face)
HF_DATASET = "junwatu/indonesian-recipes"
HF_SECRET_NAME = "HF_TOKEN"            # nama secret di Kaggle: Add-ons > Secrets
FILE_PATH = None                       # isi path lengkap jika ada beberapa file dan ingin memilih satu

# Isi manual bila deteksi otomatis di bagian 3 salah. None = biarkan otomatis.
COL_TITLE = None
COL_INGREDIENTS = None
COL_STEPS = None
COL_URL = None
COL_CATEGORY = None
COL_TIME = None
COL_DIFFICULTY = None
COL_REGION = None

SEED = 42
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working")
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Memuat dataset (read-only) dan mencatat provenance

In [ ]:
import glob, json, datetime
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
try:
    from IPython.display import display
except ImportError:
    display = print

def find_data_files():
    found = []
    for e in ("*.csv", "*.tsv", "*.json", "*.jsonl", "*.parquet"):
        found += glob.glob(os.path.join(INPUT_DIR, "**", e), recursive=True)
    return sorted(found)

def read_any(p):
    pl = p.lower()
    if pl.endswith((".csv", ".tsv")):
        sep = "\t" if pl.endswith(".tsv") else ","
        try:
            return pd.read_csv(p, sep=sep), "utf-8"
        except UnicodeDecodeError:
            print("  utf-8 gagal, mencoba latin-1 (periksa karakter rusak nanti)")
            return pd.read_csv(p, sep=sep, encoding="latin-1"), "latin-1"
    if pl.endswith(".jsonl"):
        return pd.read_json(p, lines=True), "utf-8"
    if pl.endswith(".json"):
        try:
            return pd.read_json(p), "utf-8"
        except ValueError:
            return pd.read_json(p, lines=True), "utf-8"
    if pl.endswith(".parquet"):
        return pd.read_parquet(p), "n/a"
    raise ValueError("format tidak didukung: " + p)

provenance = {
    "loaded_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "source_mode": SOURCE,
    "pandas": pd.__version__,
    "numpy": np.__version__,
}

if SOURCE == "kaggle":
    files = [FILE_PATH] if FILE_PATH else find_data_files()
    if not files:
        raise FileNotFoundError("Tidak ada file data di " + INPUT_DIR + ". Tambahkan dataset lewat Add Input.")
    if len(files) > 1:
        print("Ditemukan", len(files), "file data:")
        for f in files:
            print("  ", f)
        print("Hanya file pertama yang dimuat. Set FILE_PATH untuk memilih yang lain.")
    path = files[0]
    df_raw, enc = read_any(path)
    provenance.update({"files_found": files, "file_loaded": path, "encoding": enc,
                       "source_dataset": os.path.relpath(path, INPUT_DIR).split(os.sep)[0]})
else:
    from datasets import load_dataset
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    ds = load_dataset(HF_DATASET, token=token)
    parts = []
    for split_name, split in ds.items():
        part = split.to_pandas()
        part["hf_split"] = split_name
        parts.append(part)
    df_raw = pd.concat(parts, ignore_index=True)
    import datasets
    provenance.update({"hf_dataset": HF_DATASET, "splits": list(ds.keys()), "datasets_lib": datasets.__version__,
                       "source_dataset": HF_DATASET})

provenance["n_rows_loaded"] = int(len(df_raw))
provenance["n_cols_loaded"] = int(df_raw.shape[1])
print("Jumlah record aktual:", len(df_raw), "| kolom:", df_raw.shape[1])
df_raw.head(3)

## 3. Schema, tipe data, dan pemetaan kolom

In [ ]:
schema = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_null": df_raw.isna().sum(),
    "pct_null": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.apply(lambda s: s.astype(str).nunique()),
    "contoh_nilai": df_raw.apply(lambda s: str(s.dropna().iloc[0])[:80] if s.notna().any() else ""),
})
display(schema)

def guess_col(cols, keys, exclude=("clean", "total", "length", "count")):
    cands = [c for c in cols if not any(x in str(c).lower() for x in exclude)]
    for k in keys:
        for c in cands:
            if str(c).lower().strip() == k:
                return c
    for k in keys:
        for c in cands:
            if k in str(c).lower():
                return c
    return None

cols = list(df_raw.columns)
col = {
    "title": COL_TITLE or guess_col(cols, ["title", "judul", "nama", "name"]),
    "ingredients": COL_INGREDIENTS or guess_col(cols, ["ingredients", "bahan"]),
    "steps": COL_STEPS or guess_col(cols, ["steps", "langkah", "cara", "directions", "instructions"]),
    "url": COL_URL or guess_col(cols, ["url", "link", "source"]),
    "category": COL_CATEGORY or guess_col(cols, ["category", "kategori"]),
    "time": COL_TIME or guess_col(cols, ["time", "waktu", "duration", "durasi"]),
    "difficulty": COL_DIFFICULTY or guess_col(cols, ["difficulty", "kesulitan", "level"]),
    "region": COL_REGION or guess_col(cols, ["region", "daerah", "wilayah", "origin", "asal"]),
}
print("Pemetaan kolom (otomatis kecuali kamu isi manual). PERIKSA sendiri apakah benar:")
for k, v in col.items():
    print(f"  {k:12s} -> {v}")
print("\nKolom bernama '*clean*' sengaja diabaikan karena itu hasil olahan pembuat dataset, bukan data mentah.")
missing_core = [k for k in ("title", "ingredients", "steps") if col[k] is None]
if missing_core:
    raise ValueError("Kolom wajib tidak terdeteksi: " + str(missing_core) + ". Isi COL_* di cell konfigurasi.")

## 4. Missing values dan nilai kosong

In [ ]:
EMPTY_TOKENS = {"", "[]", "{}", "nan", "none", "null", "n/a", "-"}
def is_empty(s):
    return s.isna() | s.astype(str).str.strip().str.lower().isin(EMPTY_TOKENS)

eda = df_raw.copy()   # semua analisis di salinan; df_raw tidak disentuh
eda["_eda_title_empty"] = is_empty(eda[col["title"]])
eda["_eda_ing_empty"] = is_empty(eda[col["ingredients"]])
eda["_eda_steps_empty"] = is_empty(eda[col["steps"]])

quality = []
def add_q(name, n, note=""):
    quality.append({"pemeriksaan": name, "jumlah": int(n), "persen": round(100 * n / len(eda), 2), "catatan": note})

add_q("judul kosong/null", eda["_eda_title_empty"].sum())
add_q("bahan kosong/null", eda["_eda_ing_empty"].sum())
add_q("langkah kosong/null", eda["_eda_steps_empty"].sum())
add_q("salah satu dari ketiganya kosong", (eda["_eda_title_empty"] | eda["_eda_ing_empty"] | eda["_eda_steps_empty"]).sum())
display(pd.DataFrame(quality))

## 5. Variasi format daftar bahan dan langkah

Fungsi di bawah hanya membaca dan menebak format. Koma sengaja tidak dipakai sebagai pemisah otomatis karena ambigu ("garam, secukupnya" bisa satu bahan). Kalau banyak entri jatuh ke `single_string`, itu temuan penting untuk Phase 2.

In [ ]:
import ast, re
from collections import Counter

def to_items(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return [], "null"
    if isinstance(v, (list, tuple)):
        return [str(x) for x in v], "list_object"
    if hasattr(v, "tolist") and not isinstance(v, str):
        return [str(x) for x in v.tolist()], "array_object"
    s = str(v).strip()
    if s.lower() in EMPTY_TOKENS:
        return [], "empty"
    if s.startswith("[") and s.endswith("]"):
        for loader in (json.loads, ast.literal_eval):
            try:
                x = loader(s)
                if isinstance(x, list):
                    return [str(i) for i in x], "stringified_list"
            except Exception:
                pass
    for name, sep in (("newline", "\n"), ("double_dash", "--"), ("semicolon", ";"), ("pipe", "|")):
        if sep in s:
            return [p.strip() for p in s.split(sep) if p.strip()], name
    return [s], "single_string"

ing_parsed = eda[col["ingredients"]].map(to_items)
stp_parsed = eda[col["steps"]].map(to_items)
eda["_eda_ing_items"] = ing_parsed.map(lambda t: t[0])
eda["_eda_ing_fmt"] = ing_parsed.map(lambda t: t[1])
eda["_eda_stp_items"] = stp_parsed.map(lambda t: t[0])
eda["_eda_stp_fmt"] = stp_parsed.map(lambda t: t[1])
eda["_eda_n_ing"] = eda["_eda_ing_items"].map(len)
eda["_eda_n_stp"] = eda["_eda_stp_items"].map(len)

print("Format kolom bahan:")
display(eda["_eda_ing_fmt"].value_counts().to_frame("jumlah"))
print("Format kolom langkah:")
display(eda["_eda_stp_fmt"].value_counts().to_frame("jumlah"))

for fmt in eda["_eda_ing_fmt"].unique():
    ex = eda.loc[eda["_eda_ing_fmt"] == fmt, col["ingredients"]].head(2)
    print("\n[bahan | format =", fmt, "]")
    for e in ex:
        print("  ", repr(str(e)[:250]))

n_single = int((eda["_eda_ing_fmt"] == "single_string").sum())
add_q("bahan tidak bisa dipecah otomatis (single_string)", n_single, "perlu aturan pemisah khusus di Phase 2")

## 6. Duplikasi (non-destruktif)

In [ ]:
import unicodedata

def norm_title(t):
    t = unicodedata.normalize("NFKC", str(t)).lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

eda["_eda_title_norm"] = eda[col["title"]].map(norm_title)
key_cols = [col["title"], col["ingredients"], col["steps"]]
n_exact = int(eda.astype({c: str for c in key_cols}).duplicated(subset=key_cols, keep="first").sum())
add_q("duplikat persis (judul+bahan+langkah)", n_exact, "baris ke-2 dst dari kelompok yang sama")

nonempty_t = eda[~eda["_eda_title_empty"]]
dup_title_mask = nonempty_t.duplicated(subset="_eda_title_norm", keep=False)
n_dup_title_rows = int(dup_title_mask.sum())
n_dup_title_groups = int(nonempty_t.loc[dup_title_mask, "_eda_title_norm"].nunique())
add_q("baris dengan judul ternormalisasi yang sama", n_dup_title_rows, f"{n_dup_title_groups} kelompok judul")

# Dari kelompok judul sama: berapa yang bahannya juga sama (kandidat near-duplicate) vs berbeda (resep beda, judul sama)
tmp = nonempty_t.loc[dup_title_mask].copy()
tmp["_eda_ing_key"] = tmp["_eda_ing_items"].map(lambda xs: "|".join(sorted(x.strip().lower() for x in xs)))
grp = tmp.groupby("_eda_title_norm")["_eda_ing_key"].nunique()
n_same_title_diff_ing = int((grp > 1).sum())
n_same_title_same_ing = int((grp == 1).sum())
add_q("kelompok judul sama, bahan juga identik", n_same_title_same_ing, "kandidat near-duplicate")
add_q("kelompok judul sama, bahan berbeda", n_same_title_diff_ing, "kemungkinan resep berbeda dengan judul sama")
display(pd.DataFrame(quality))

if n_dup_title_groups:
    top_dups = nonempty_t.loc[dup_title_mask, "_eda_title_norm"].value_counts().head(10)
    print("Judul paling sering berulang:")
    display(top_dups.to_frame("jumlah"))

## 7. Distribusi jumlah bahan dan langkah

In [ ]:
import matplotlib.pyplot as plt

valid = eda[~eda["_eda_ing_empty"]]
stats_ing = valid["_eda_n_ing"].describe(percentiles=[.05, .25, .5, .75, .95]).round(2)
valid_s = eda[~eda["_eda_steps_empty"]]
stats_stp = valid_s["_eda_n_stp"].describe(percentiles=[.05, .25, .5, .75, .95]).round(2)
display(pd.DataFrame({"jumlah_bahan": stats_ing, "jumlah_langkah": stats_stp}))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(valid["_eda_n_ing"].clip(upper=60), bins=40)
ax[0].set_title("Jumlah bahan per resep (dipotong di 60)")
ax[1].hist(valid_s["_eda_n_stp"].clip(upper=40), bins=40)
ax[1].set_title("Jumlah langkah per resep (dipotong di 40)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "eda_distribusi.png"), dpi=120)
plt.show()

print("Catatan: kalau 'jumlah langkah' hampir selalu 1, kemungkinan langkah tersimpan sebagai satu paragraf, bukan daftar.")

## 8. Frekuensi bahan (kasar, BUKAN normalisasi)

Pembersihan di sini sangat sederhana: ambil teks sebelum koma atau kurung, buang angka dan satuan di depan. Varian ejaan seperti "telor" dan "telur" akan muncul sebagai entri terpisah. Itu bukan bug, itu bukti kebutuhan normalisasi di Phase 3.

In [ ]:
UNITS = ["sendok makan", "sendok teh", "sdm", "sdt", "siung", "buah", "biji", "batang", "lembar", "gram", "gr", "kg",
         "ml", "liter", "ruas", "butir", "bungkus", "sachet", "secukupnya", "cm", "potong", "ikat", "genggam",
         "gelas", "cangkir", "g", "l"]
unit_re = re.compile(r"^(" + "|".join(sorted(UNITS, key=len, reverse=True)) + r")\b\.?\s*")

def rough_name(line):
    s = str(line).lower()
    s = re.split(r"[,(]", s)[0]
    s = re.sub(r"^[\s\-\u2022\*\d\.\/\u00bd\u00bc\u00be]+", "", s)
    s = unit_re.sub("", s)
    return s.strip()

cnt = Counter()
n_items = 0
for items in valid["_eda_ing_items"]:
    for it in items:
        n = rough_name(it)
        if n:
            cnt[n] += 1
        n_items += 1
print("Total baris bahan diproses:", n_items, "| nama kasar unik:", len(cnt))
top = pd.DataFrame(cnt.most_common(50), columns=["nama_kasar", "frekuensi"])
display(top)

fig, ax = plt.subplots(figsize=(8, 6))
t30 = top.head(30).iloc[::-1]
ax.barh(t30["nama_kasar"], t30["frekuensi"])
ax.set_title("30 bahan teratas (kasar, belum dinormalisasi)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "eda_top_bahan.png"), dpi=120)
plt.show()

singles = sum(1 for v in cnt.values() if v == 1)
add_q("nama bahan kasar yang hanya muncul sekali", singles, "indikator ekor panjang/variasi ejaan/noise")

## 9. Sampel representatif dan sampel bermasalah

In [ ]:
def show_samples(frame, n, title):
    print("=" * 10, title, "=" * 10)
    for idx, row in frame.head(n).iterrows():
        print(f"[{idx}] {str(row[col['title']])[:80]}")
        print("    bahan  (%d):" % row["_eda_n_ing"], [x[:40] for x in row["_eda_ing_items"][:6]])
        print("    langkah(%d):" % row["_eda_n_stp"], (row["_eda_stp_items"][0][:100] if row["_eda_stp_items"] else "-"))

sample_df = eda.sample(n=min(10, len(eda)), random_state=SEED)
show_samples(sample_df, 10, "10 SAMPEL ACAK (seed=%d)" % SEED)

problems = {
    "judul kosong": eda["_eda_title_empty"],
    "bahan kosong": eda["_eda_ing_empty"],
    "langkah kosong": eda["_eda_steps_empty"],
    "bahan <= 2 item": (eda["_eda_n_ing"] <= 2) & ~eda["_eda_ing_empty"],
    "bahan >= 40 item": eda["_eda_n_ing"] >= 40,
    "judul sangat pendek (<3 karakter)": eda[col["title"]].astype(str).str.len() < 3,
}
prob_rows = []
for name, mask in problems.items():
    prob_rows.append({"jenis_masalah": name, "jumlah": int(mask.sum())})
    add_q("[bermasalah] " + name, mask.sum())
display(pd.DataFrame(prob_rows))
for name, mask in problems.items():
    if mask.sum() > 0:
        show_samples(eda[mask].sample(n=min(3, int(mask.sum())), random_state=SEED), 3, "MASALAH: " + name)

## 10. Pemetaan ke schema target

In [ ]:
def status(c):
    return ("tersedia: " + str(c)) if c else "TIDAK tersedia"

mapping = pd.DataFrame([
    ("recipe_id", "dapat dibuat (indeks sintetis), bukan dari sumber"),
    ("title", status(col["title"])),
    ("ingredients_raw", status(col["ingredients"])),
    ("ingredients_normalized", "belum ada, dikerjakan Phase 3"),
    ("ingredients_structured", "belum ada, dikerjakan Phase 2-3"),
    ("steps", status(col["steps"])),
    ("category", status(col["category"])),
    ("difficulty", status(col["difficulty"])),
    ("time_minutes", status(col["time"])),
    ("region", status(col["region"])),
    ("source_url", status(col["url"])),
    ("source_dataset", "dicatat dari provenance"),
    ("duplicate_group_id", "dapat diturunkan dari bagian 6, belum dibuat"),
], columns=["field_target", "status"])
display(mapping)

# Cek isi metadata yang kolomnya ada: bisa saja kolom ada tetapi hampir kosong.
for k in ("category", "difficulty", "time", "region", "url"):
    if col[k]:
        filled = (~is_empty(eda[col[k]])).mean() * 100
        print(f"  {k}: terisi {filled:.1f}% | contoh: {eda[col[k]].dropna().astype(str).head(3).tolist()}")

## 11. Simpan laporan dan provenance

In [ ]:
quality_df = pd.DataFrame(quality)
quality_df.to_csv(os.path.join(OUT_DIR, "data_quality.csv"), index=False)
mapping.to_csv(os.path.join(OUT_DIR, "schema_mapping.csv"), index=False)
sample_df[[col["title"], col["ingredients"], col["steps"]]].to_csv(os.path.join(OUT_DIR, "sample_records.csv"), index=False)
top.to_csv(os.path.join(OUT_DIR, "top_ingredients_rough.csv"), index=False)

provenance["column_mapping"] = {k: (str(v) if v else None) for k, v in col.items()}
provenance["n_records"] = int(len(eda))
provenance["license_note"] = "BELUM DIVERIFIKASI. Isi manual setelah membaca halaman dataset."
with open(os.path.join(OUT_DIR, "provenance.json"), "w", encoding="utf-8") as f:
    json.dump(provenance, f, indent=2, ensure_ascii=False, default=str)

report = {
    "provenance": provenance,
    "quality": quality,
    "ingredient_formats": eda["_eda_ing_fmt"].value_counts().to_dict(),
    "step_formats": eda["_eda_stp_fmt"].value_counts().to_dict(),
    "n_ingredients_stats": {k: float(v) for k, v in stats_ing.items()},
    "n_steps_stats": {k: float(v) for k, v in stats_stp.items()},
}
with open(os.path.join(OUT_DIR, "eda_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)

print("Tersimpan di", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print(" ", f)
display(quality_df)

## 12. Lisensi dan provenance (ISI MANUAL)

Buka halaman sumber dataset, baca ketentuannya, lalu isi:

- URL sumber:
- Tanggal akses:
- Lisensi yang tertulis di halaman:
- Apakah boleh dipakai untuk tugas kuliah/riset:
- Apakah boleh didistribusikan ulang:
- Catatan lain:

Jika lisensi tidak tertulis, tulis "tidak tertulis". Jangan menebak.

## 13. Kesimpulan Phase 1 (ISI MANUAL setelah membaca output di atas)

Jawab dengan angka dari output, bukan perkiraan:

1. Jumlah record aktual dan jumlah yang layak pakai (ketiga kolom inti terisi):
2. Format daftar bahan: bisa dipecah otomatis atau tidak? Pemisah apa?
3. Persentase duplikat persis dan kandidat near-duplicate:
4. Metadata apa yang benar-benar terisi (waktu, kategori, kesulitan, wilayah)?
5. Masalah data terbesar yang akan menyulitkan Phase 2:
6. Keputusan: dataset ini dipakai / tidak dipakai, alasannya: